# 01 - Data Preprocessing: CIC-IDS2017
   Combines all 8 daily CSV files, applies binary labeling (BENIGN vs ANOMALY), 
   and handles data quality issues (infinities, NaNs, duplicates).

In [1]:
import sys
print(sys.executable)

/home/vboxuser/FEAR/fear-env/bin/python3


In [4]:
import pandas as pd
df = pd.read_csv('~/FEAR/data/cicids2017/MachineLearningCSV/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv')
df.shape

(529918, 79)

In [5]:
print(df.shape)
print(df.columns.tolist())

(529918, 79)
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count'

In [8]:
print(df[' Label'].value_counts())

 Label
BENIGN    529918
Name: count, dtype: int64


In [9]:
df_tue = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017/MachineLearningCSV/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv')
print(df_tue.shape)
print(df_tue[' Label'].value_counts())

(445909, 79)
 Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64


In [10]:
df_wed = pd.read_csv('/home/vboxuser/FEAR/data/cicids2017/MachineLearningCSV/MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv')
print(df_wed.shape)
print(df_wed[' Label'].value_counts())

(692703, 79)
 Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [12]:
import glob
path = '/home/vboxuser/FEAR/data/cicids2017/MachineLearningCSV/MachineLearningCVE/'
all_files = glob.glob(path + '*.csv')
print(len(all_files))  # should print 8
df_list = [pd.read_csv(f) for f in all_files]
df_all = pd.concat(df_list, ignore_index=True)
print(df_all.shape)

8
(2830743, 79)


In [14]:
df_all.columns = df_all.columns.str.strip()
print(df_all.columns.tolist()[:5])  # quick check the spaces are gone

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets']


In [15]:
df_all['Binary_Label'] = df_all['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
print(df_all['Binary_Label'].value_counts())

Binary_Label
0    2273097
1     557646
Name: count, dtype: int64


In [ ]:
#2,273,097 BENIGN (80.3%) vs. 557,646 ANOMALY (19.7%) ~80/20 benign-to-anomaly split across 2.83M flows."

In [16]:
import numpy as np
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df_all.isna().sum().sum())

5734


In [17]:
df_all.dropna(inplace=True)
print(df_all.shape)

(2827876, 80)


In [18]:
print(df_all.duplicated().sum())

307078


In [19]:
df_all.drop_duplicates(inplace=True)
print(df_all.shape)

(2520798, 80)


In [20]:
df_all.to_csv('/home/vboxuser/FEAR/data/cicids2017_cleaned.csv', index=False)